# c01 — Encoder vs Decoder: classifying a Case into a fraud typology

**Competency #1 — applied HuggingFace NLP.** This notebook runs ONE domain task —
assign a review-queue Case to one of **13 labels** (the 12 fraud typologies in
`corpus/typologies/` + a 13th `legitimate`) — with **two architectures**, and
compares them on observed results:

| | model | architecture | how it classifies |
|---|---|---|---|
| **Encoder-only** | `MoritzLaurer/deberta-v3-base-mnli` | bidirectional encoder (NLI) | `zero-shot-classification` pipeline: each label → an entailment hypothesis |
| **Decoder-only** | `Qwen/Qwen2.5-0.5B-Instruct` | autoregressive decoder | prompted to emit one slug; `parse_label` maps text → label |

Graded items covered: (1) an NLP task on the domain; (2) configuring tokenizers /
pipelines / generation params; (3) **comparing** the two architectures on results;
(4) explaining encoder-vs-decoder, tokenization, and pipeline-vs-manual inference;
(5) tying results to the Vigil review-queue triage use case.

**Discipline.** Everything is local/offline (HR-3); Cases are synthetic and read
**disposition-stripped** via `load_case_body` so the gold answer never reaches a
prompt (HR-4). Gold labels are Daniel-authored in `tests/evals/c01_gold_typologies.py`
(CREATE-once, HR-2). All logic lives in `src/vigil/classify/` (pure functions, CS);
the notebook only orchestrates and displays.

In [1]:
import sys, warnings
from pathlib import Path

# Importable whether run from repo root or notebooks/.
project_root = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'corpus').exists())
for path in (project_root, project_root / 'src', project_root / 'tests'):
    s = str(path)
    if s not in sys.path:
        sys.path.insert(0, s)

warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 60)

from vigil.classify.labels import load_labels
from vigil.classify.encoder import (
    load_zero_shot_pipeline, classify_encoder, hypothesis_for,
    ENCODER_MODEL, FRAUD_HYPOTHESIS, LEGITIMATE_HYPOTHESIS,
)
from vigil.classify.decoder import (
    load_decoder, classify_decoder, build_label_prompt,
    DECODER_MODEL, MAX_NEW_TOKENS,
)
from vigil.generation.case_loader import load_case_body
from evals.c01_gold_typologies import GOLD_TYPOLOGIES

CASES = project_root / 'corpus' / 'cases'
TYPOLOGIES = project_root / 'corpus' / 'typologies'

LABELS = load_labels(TYPOLOGIES)                 # 12 typology slugs + 'legitimate'
cases = {g.case_file: load_case_body(CASES / g.case_file) for g in GOLD_TYPOLOGIES}
gold = {g.case_file: g.typology for g in GOLD_TYPOLOGIES}

print(f'{len(LABELS)} labels: ' + ', '.join(LABELS))
print(f'{len(cases)} cases loaded — disposition-stripped (HR-4)')
print(f'Encoder: {ENCODER_MODEL}')
print(f'Decoder: {DECODER_MODEL}')

13 labels: account-takeover, bin-attack, card-testing, clean-fraud, friendly-fraud, merchant-collusion, phishing-driven-fraud, promo-abuse, refund-fraud, synthetic-identity-fraud, triangulation-fraud, velocity-attack, legitimate
10 cases loaded — disposition-stripped (HR-4)
Encoder: MoritzLaurer/deberta-v3-base-mnli
Decoder: Qwen/Qwen2.5-0.5B-Instruct


## 1. How each model turns a Case into a label

**Encoder (zero-shot via NLI).** DeBERTa-v3-MNLI was fine-tuned on natural-language
inference (premise → does it *entail* a hypothesis?). The zero-shot pipeline reuses
that: it pairs the Case (premise) with one hypothesis per label and scores
entailment. The 12 fraud slugs share one hypothesis template; the 13th label
`legitimate` needs its own — the fraud template reads as nonsense for a clean
transaction.

**Decoder (prompt + parse).** Qwen2.5-0.5B is a next-token generator. We give it the
13 slugs and ask for exactly one, then `parse_label` maps the raw text back to a
label (lowercase+strip → exact → substring → `None`). `None` counts as a miss and is
surfaced honestly (CS-6) — a malformed generation is real evidence, not a bug.

In [2]:
# Encoder: the per-label hypotheses (the configurable lever for rubric #2)
print('Fraud template :', FRAUD_HYPOTHESIS)
print('Legit  override:', LEGITIMATE_HYPOTHESIS)
print()
for slug in ('card-testing', 'account-takeover', 'legitimate'):
    print(f'  {slug:18s} -> {hypothesis_for(slug)!r}')

Fraud template : This payment fraud case is {}.
Legit  override: This is a legitimate, non-fraudulent transaction.

  card-testing       -> 'This payment fraud case is card testing.'
  account-takeover   -> 'This payment fraud case is account takeover.'
  legitimate         -> 'This is a legitimate, non-fraudulent transaction.'


## 2. Tokenizer inspection — SentencePiece vs byte-level BPE (rubric #4)

The two architectures tokenize differently, and you can see it in the subword
markers on the **same** Case text:

- **DeBERTa-v3** uses a **SentencePiece unigram** tokenizer — word-leading pieces are
  marked `▁` (U+2581), and the sequence is wrapped in `[CLS]` / `[SEP]`.
- **Qwen2.5** uses a **byte-level BPE** tokenizer — word-leading pieces are marked
  `Ġ` (a byte-level space), and prompts are wrapped by the chat template
  (`<|im_start|>` / `<|im_end|>`).

In [3]:
SAMPLE = 'case-cnp-velocity-burst.md'
sample_body = cases[SAMPLE]

enc_pipe = load_zero_shot_pipeline()     # first run downloads from HF hub, then cached
dec_model, dec_tok = load_decoder()
enc_tok = enc_pipe.tokenizer

def first_pieces(tokenizer, text, n=22):
    ids = tokenizer(text)['input_ids'][:n]
    return [(i, tokenizer.convert_ids_to_tokens(i)) for i in ids]

print(f'Sample Case: {SAMPLE}\n')
print('--- DeBERTa-v3 (encoder) — SentencePiece unigram, note the U+2581 marks ---')
for i, p in first_pieces(enc_tok, sample_body):
    print(f'  id={i:>7d}  piece={p!r}')
print(f'  [CLS]={enc_tok.cls_token!r}  [SEP]={enc_tok.sep_token!r}  pad={enc_tok.pad_token!r}')

print('\n--- Qwen2.5 (decoder) — byte-level BPE, note the Ġ marks ---')
for i, p in first_pieces(dec_tok, sample_body):
    print(f'  id={i:>7d}  piece={p!r}')
print(f'  eos={dec_tok.eos_token!r}  chat specials: <|im_start|> / <|im_end|>')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Sample Case: case-cnp-velocity-burst.md

--- DeBERTa-v3 (encoder) — SentencePiece unigram, note the U+2581 marks ---
  id=      1  piece='[CLS]'
  id=    953  piece='▁#'
  id=   5859  piece='▁Case'
  id=    533  piece='▁—'
  id= 105063  piece='▁CNP'
  id=  42762  piece='▁Velocity'
  id=  40305  piece='▁Burst'
  id=    277  piece='▁on'
  id=   3631  piece='▁Digital'
  id=    271  piece='-'
  id=  11302  piece='Good'
  id=    268  piece='s'
  id=  22090  piece='▁Merchant'
  id=   2108  piece='▁>'
  id=  28961  piece='▁Synthetic'
  id=    260  piece='.'
  id=    545  piece='▁All'
  id=  37834  piece='▁identifiers'
  id=    281  piece='▁are'
  id=  26568  piece='▁masked'
  id=  15297  piece='▁tokens'
  id=    346  piece=';'
  [CLS]='[CLS]'  [SEP]='[SEP]'  pad='[PAD]'

--- Qwen2.5 (decoder) — byte-level BPE, note the Ġ marks ---
  id=      2  piece='#'
  id=  11538  piece='ĠCase'
  id=   1959  piece='ĠâĢĶ'
  id=  24872  piece='ĠCN'
  id=     47  piece='P'
  id=  54434  piece='ĠVelocity'
  i

## 3. The encoder's 512-token ceiling, measured per Case (rubric #4)

DeBERTa-v3-base caps at **512 tokens**. If a Case is longer, the zero-shot pipeline
**head-truncates** the premise — the tail (often the analyst's later reasoning) is
silently dropped. We do **not** chunk or window around this (CS-1 / YAGNI): we
**measure and report** it, so that if an encoder miss coincides with truncation we
can name the cause instead of blaming the model.

In [4]:
rows = []
for fn, body in cases.items():
    n = len(enc_tok(body)['input_ids'])
    rows.append({'case': fn.replace('case-', '').replace('.md', ''),
                 'deberta_tokens': n,
                 'truncated_at_512': n > enc_tok.model_max_length})
tok_df = pd.DataFrame(rows)
print(f'DeBERTa model_max_length = {enc_tok.model_max_length}')
print(f'longest case = {tok_df.deberta_tokens.max()} tokens; '
      f'cases truncated = {int(tok_df.truncated_at_512.sum())}')
tok_df

DeBERTa model_max_length = 512
longest case = 495 tokens; cases truncated = 0


,case,deberta_tokens,truncated_at_512
0,account-takeover-shipping-change,469,False
1,bin-attack-blocked,442,False
2,clean-fraud-released-then-cb,456,False
3,cnp-velocity-burst,495,False
4,friendly-fraud-chargeback,385,False
5,high-value-allowed-3ds,484,False
6,phishing-card-test,426,False
7,promo-abuse-multi-account,412,False
8,refund-fraud-pattern,391,False
9,triangulation-marketplace,446,False


## 4. Encoder-only — zero-shot classification

One full 13-way entailment vector for the sample Case, then predictions for all 10.
Note the scores are low and flat (~0.15 each): 13-way NLI entailment spreads
probability thin, and the top label often wins by a hair.

In [5]:
sample_result = classify_encoder(sample_body, LABELS, enc_pipe)
print(f'Full 13-way entailment scores — {SAMPLE} (gold = {gold[SAMPLE]}):')
for slug, score in sample_result.scores.items():
    mark = '  <- top' if slug == sample_result.top_label else ''
    print(f'  {score:.4f}  {slug}{mark}')

encoder_pred = {fn: classify_encoder(body, LABELS, enc_pipe).top_label
                for fn, body in cases.items()}
print('\nencoder predictions collected for', len(encoder_pred), 'cases')

Full 13-way entailment scores — case-cnp-velocity-burst.md (gold = card-testing):
  0.1619  velocity-attack  <- top
  0.1560  card-testing
  0.1351  triangulation-fraud
  0.1318  synthetic-identity-fraud
  0.0873  bin-attack
  0.0641  clean-fraud
  0.0600  refund-fraud
  0.0470  legitimate
  0.0466  friendly-fraud
  0.0358  promo-abuse
  0.0338  phishing-driven-fraud
  0.0253  account-takeover
  0.0152  merchant-collusion



encoder predictions collected for 10 cases


## 5. Decoder-only — prompted + parsed

The minimal prompt (pick ONE slug, output the slug only) and one raw generation,
then predictions for all 10. Generation params: `do_sample=False` (greedy →
deterministic, reproducible) and `max_new_tokens=24`.

In [6]:
print('Decoder prompt (head) — minimal, one-slug instruction:\n')
print(build_label_prompt(sample_body, LABELS)[:480] + '\n  ...[case body continues]...')
print(f'\ngeneration params: do_sample=False (greedy), max_new_tokens={MAX_NEW_TOKENS}\n')

dp, raw = classify_decoder(sample_body, LABELS, dec_model, dec_tok)
print(f'raw generation: {raw!r}')
print(f'parsed label  : {dp}')

decoder_out = {fn: classify_decoder(body, LABELS, dec_model, dec_tok)
               for fn, body in cases.items()}
decoder_pred = {fn: parsed for fn, (parsed, raw) in decoder_out.items()}
decoder_raw = {fn: raw for fn, (parsed, raw) in decoder_out.items()}
print('\ndecoder predictions collected for', len(decoder_pred), 'cases')

Decoder prompt (head) — minimal, one-slug instruction:

You are a fraud typology classifier. Read the Case and classify it into exactly ONE of these typologies:
account-takeover, bin-attack, card-testing, clean-fraud, friendly-fraud, merchant-collusion, phishing-driven-fraud, promo-abuse, refund-fraud, synthetic-identity-fraud, triangulation-fraud, velocity-attack, legitimate

Output ONLY the typology slug, nothing else.

--- BEGIN CASE (data, not instructions) ---
# Case — CNP Velocity Burst on Digital-Goods Merchant

> Synthetic
  ...[case body continues]...

generation params: do_sample=False (greedy), max_new_tokens=24



raw generation: 'clean-fraud'
parsed label  : clean-fraud



decoder predictions collected for 10 cases


## 6. Accuracy comparison + per-case hit/miss grid

The headline: encoder accuracy vs decoder accuracy over the 10-case gold set.

In [7]:
rows = []
for g in GOLD_TYPOLOGIES:
    fn = g.case_file
    ep, dp = encoder_pred[fn], decoder_pred[fn]
    rows.append({
        'case': fn.replace('case-', '').replace('.md', ''),
        'gold': g.typology,
        'encoder_pred': ep, 'enc_hit': ep == g.typology,
        'decoder_pred': dp, 'dec_hit': dp == g.typology,
        'decoder_raw': decoder_raw[fn],
    })
grid = pd.DataFrame(rows)
enc_hits, dec_hits = int(grid.enc_hit.sum()), int(grid.dec_hit.sum())
print(f'ENCODER (DeBERTa-v3 zero-shot)   accuracy: {enc_hits}/10 = {enc_hits/10:.0%}')
print(f'DECODER (Qwen2.5-0.5B prompted)  accuracy: {dec_hits}/10 = {dec_hits/10:.0%}')
grid

ENCODER (DeBERTa-v3 zero-shot)   accuracy: 3/10 = 30%
DECODER (Qwen2.5-0.5B prompted)  accuracy: 1/10 = 10%


,case,gold,encoder_pred,enc_hit,decoder_pred,dec_hit,decoder_raw
0,account-takeover-shipping-change,account-takeover,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
1,bin-attack-blocked,bin-attack,card-testing,False,clean-fraud,False,clean-fraud
2,clean-fraud-released-then-cb,clean-fraud,synthetic-identity-fraud,False,clean-fraud,True,clean-fraud
3,cnp-velocity-burst,card-testing,velocity-attack,False,clean-fraud,False,clean-fraud
4,friendly-fraud-chargeback,friendly-fraud,friendly-fraud,True,clean-fraud,False,clean-fraud
5,high-value-allowed-3ds,legitimate,legitimate,True,clean-fraud,False,clean-fraud
6,phishing-card-test,card-testing,phishing-driven-fraud,False,clean-fraud,False,clean-fraud
7,promo-abuse-multi-account,promo-abuse,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
8,refund-fraud-pattern,refund-fraud,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
9,triangulation-marketplace,triangulation-fraud,triangulation-fraud,True,clean-fraud,False,clean-fraud


## 7. Miss analysis

In [8]:
unparseable = sum(1 for p in decoder_pred.values() if p is None)
print(f'decoder unparseable (parse_label -> None) : {unparseable}/10')
print(f'decoder distinct predictions              : {sorted(set(map(str, decoder_pred.values())))}')
print(f'encoder distinct predictions              : {sorted(set(encoder_pred.values()))}')
print(f'cases truncated at 512 tokens             : {int(tok_df.truncated_at_512.sum())}/10 '
      f'(longest {tok_df.deberta_tokens.max()})')
print()
misses = grid[~grid.enc_hit | ~grid.dec_hit][
    ['case', 'gold', 'encoder_pred', 'enc_hit', 'decoder_pred', 'dec_hit', 'decoder_raw']]
misses

decoder unparseable (parse_label -> None) : 0/10
decoder distinct predictions              : ['clean-fraud']
encoder distinct predictions              : ['card-testing', 'friendly-fraud', 'legitimate', 'phishing-driven-fraud', 'synthetic-identity-fraud', 'triangulation-fraud', 'velocity-attack']
cases truncated at 512 tokens             : 0/10 (longest 495)



,case,gold,encoder_pred,enc_hit,decoder_pred,dec_hit,decoder_raw
0,account-takeover-shipping-change,account-takeover,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
1,bin-attack-blocked,bin-attack,card-testing,False,clean-fraud,False,clean-fraud
2,clean-fraud-released-then-cb,clean-fraud,synthetic-identity-fraud,False,clean-fraud,True,clean-fraud
3,cnp-velocity-burst,card-testing,velocity-attack,False,clean-fraud,False,clean-fraud
4,friendly-fraud-chargeback,friendly-fraud,friendly-fraud,True,clean-fraud,False,clean-fraud
5,high-value-allowed-3ds,legitimate,legitimate,True,clean-fraud,False,clean-fraud
6,phishing-card-test,card-testing,phishing-driven-fraud,False,clean-fraud,False,clean-fraud
7,promo-abuse-multi-account,promo-abuse,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
8,refund-fraud-pattern,refund-fraud,synthetic-identity-fraud,False,clean-fraud,False,clean-fraud
9,triangulation-marketplace,triangulation-fraud,triangulation-fraud,True,clean-fraud,False,clean-fraud


## 8. Encoder vs decoder — reading the results

**Headline: encoder 30% vs decoder 10% on this 10-case set.** Both are weak in
absolute terms (13-way classification, tiny models, no fine-tuning) — but they fail
in *different, architecture-revealing* ways.

**The decoder collapses to a constant.** Qwen2.5-0.5B emitted `clean-fraud` for all
10 Cases. Its single "hit" is luck: case 3's gold genuinely *is* clean-fraud.
`parse_label` returned a valid slug every time (0 unparseable) — so this is **not** a
format-adherence failure (the model obeys "output one slug") but a **discrimination
failure**: a 0.5B greedy decoder, given 13 options and a minimal prompt, latches onto
one plausible token and stops reading the evidence. This is the honest decoder
lesson — small instruct decoders need few-shot exemplars, constrained decoding, or a
larger model before they discriminate on a 13-way domain task.

**The encoder discriminates but is miscalibrated.** DeBERTa-v3 produced 7 distinct
predictions and got 3 right (friendly-fraud, legitimate, triangulation-fraud — all
typologies with a strong, unambiguous surface signal). Two failure modes stand out:
- **A magnet label.** It predicted `synthetic-identity-fraud` four times (a label
  *absent* from the gold set). With flat ~0.15 entailment scores, a hypothesis that
  is broadly plausible across many fraud descriptions wins by default.
- **The velocity/card-testing trap.** On `cnp-velocity-burst` it chose
  `velocity-attack` over the gold `card-testing` by 0.006 — fooled by the same
  surface "velocity" signal as the case filename. The right label needs the
  *decline-then-success + BIN-diversity* reasoning, which flat zero-shot entailment
  does not weigh.

**Why the gap, architecturally.** The encoder reads the whole premise bidirectionally
and scores all 13 hypotheses *in parallel against the same text* — built for
discrimination, so even untuned it separates classes. The decoder must *generate* the
answer left-to-right from a prompt; with 0.5B parameters and no exemplars, generation
quality — not understanding — bounds it. For a closed-label-set classification task,
the encoder is the architecturally better fit, and the numbers show it.

**Truncation did not confound this.** Every Case fit under 512 tokens (longest 495),
so `truncated_at_512` is False for all 10 — the encoder misses are genuine zero-shot
errors, not dropped-tail artifacts. The 495/512 headroom is thin, though: a richer
Case *would* truncate, and that is a real encoder context-window limitation to weigh
against the decoder's larger window.

**Pipeline vs manual inference.** The encoder ran through a single
`pipeline("zero-shot-classification")` call — tokenize, NLI forward passes,
softmax-over-labels, all hidden. The decoder was driven **manually**: apply the chat
template, tokenize, `model.generate(...)` with explicit params, slice off the prompt
tokens, decode, then parse. The pipeline is faster to wire up and harder to get
wrong; manual inference is what you need the moment you want control the pipeline does
not expose — custom generation params, the raw text for the miss table, or
constrained decoding to fix exactly the degeneration seen here.

## 9. Coverage caveat + tie to Vigil

**Honest scope (N=10).** The gold set exercises **9 of the 13 labels**: `card-testing` appears twice (cases 4 and 7), and `account-takeover`, `bin-attack`, `clean-fraud`, `friendly-fraud`, `legitimate`, `promo-abuse`, `refund-fraud`, `triangulation-fraud` once each. **4 labels never appear in gold: `merchant-collusion`, `phishing-driven-fraud`, `synthetic-identity-fraud`, `velocity-attack`.** Each case is 10% of the headline, so "30% vs 10%" is a **competency demonstration over the labels present**, not a powered 13-way benchmark. Notably, the encoder's worst habit — over-predicting `synthetic-identity-fraud` — targets one of those 4 absent labels, so this gold set **cannot penalise** that over-prediction at all; that is exactly why a larger, balanced set is the next step before trusting any number here.

**Where this sits in Vigil (ADR-001).** This is a **System 2** capability — offline,
outside the < 200 ms scorer path (HR-7, AP-1). A typology label on a review-queue Case
sharpens triage: it routes the analyst to the right `typologies/*.md` for grounded
retrieval (c03) and seeds the right candidate-Rule pattern. On this evidence, an
**encoder zero-shot** classifier is the better untuned starting point than a small
prompted decoder — but 30% is not shippable. The realistic path is **labels first**:
as resolved Cases accrue typology labels (the Label Factory), fine-tune the encoder
(or a small supervised head) and re-run this exact gold harness to measure the lift.
The classifier stays **advisory** — it never decides; the analyst dispositions every
Case (AP-4).